# Plan using API Kemendikti to get detail
try using cloudscraper to handle firewall


In [1]:
pip install requests

Note: you may need to restart the kernel to use updated packages.


In [14]:
pip install cloudscraper

Note: you may need to restart the kernel to use updated packages.


In [20]:
import cloudscraper
import requests
import json

scraper = cloudscraper.create_scraper()

def get_school_details(npsn):
    """Fetches details for a specific NPSN including latitude and longitude."""
    detail_url = f"https://api.data.belajar.id/data-portal-backend/v1/master-data/satuan-pendidikan/details/{npsn}"
    try:
        time.sleep(0.5)
        response = scraper.get(detail_url, timeout=15)
        
        if response.status_code == 200:
            sp = response.json().get("satuanPendidikan", {})
            return {
                "npsn": sp.get("npsn"),
                "lat": sp.get("lintang"),
                "lon": sp.get("bujur"),
                "alamat": sp.get("alamatJalan")
            }
        else:
            print(f"Failed NPSN {npsn}: Status {response.status_code}")
            return None
    except Exception as e:
        print(f"Error fetching NPSN {npsn}: {e}")
        return None

In [16]:
import sqlite3
import pandas as pd
import time

file_path_csv = "data_induk_satuan_pendidikan-daftar_nasional-23_februari_2026.csv"

In [18]:
df = pd.read_csv(file_path_csv, skiprows=1)
df

,NPSN,Nama,Bentuk,Jenis,Status,Jenjang,Kabupaten,Kecamatan,Kelurahan,Alamat,Jalur,Pembina
0,69797064,KB AL - JIHAD,KB,-,SWASTA,PENDIDIKAN ANAK USIA DINI,KAB. MOJOKERTO,KEC. GEDEG,BERATWETAN,JL. H. ABDUL FATAH NO. 17,NON FORMAL,KEMENTERIAN PENDIDIKAN DASAR DAN MENENGAH
1,69738469,RA/BA/TA ALFALAQIAH,RA,-,SWASTA,PENDIDIKAN ANAK USIA DINI,KOTA BOGOR,KEC. KOTA BOGOR BARAT,-,PAGENTONGAN,NON FORMAL,KEMENTERIAN AGAMA
2,69821591,KB MANARUL HUDA,KB,-,SWASTA,PENDIDIKAN ANAK USIA DINI,KAB. TASIKMALAYA,KEC. CULAMEGA,CIKUYA,KP. CILANJUNG,NON FORMAL,KEMENTERIAN PENDIDIKAN DASAR DAN MENENGAH
3,20533506,SDN KALIASIN VII/ 286,SD,PENDIDIKAN UMUM,NEGERI,PENDIDIKAN DASAR,KOTA SURABAYA,KEC. GENTENG,EMBONG KALIASIN,JL. EMBONG BLIMBING NO. 40,FORMAL,KEMENTERIAN PENDIDIKAN DASAR DAN MENENGAH
4,60720825,MIN 2 KOTA PASURUAN,MI,PENDIDIKAN UMUM,NEGERI,PENDIDIKAN DASAR,KOTA PASURUAN,KEC. BUGUL KIDUL,BUGULKIDUL,JL. NANAS RAYA PERUMNAS BUGUL PERMAI,FORMAL,KEMENTERIAN AGAMA
...,...,...,...,...,...,...,...,...,...,...,...,...
551973,P9999984,PKBM WAHANA WACANA,PKBM,-,SWASTA,PENDIDIKAN KEMASYARAKATAN,KAB. NABIRE,KEC. YARO,WIRASKA,JL. NABIRE WANGGAR WIRASKA JALUR 1,NON FORMAL,KEMENTERIAN PENDIDIKAN DASAR DAN MENENGAH
551974,60714164,MI GUPPI WIDORO,MI,PENDIDIKAN UMUM,SWASTA,PENDIDIKAN DASAR,KAB. PACITAN,KEC. DONOROJO,WIDORO,DESA WIDORO,FORMAL,KEMENTERIAN AGAMA
551975,20277084,MAS YPP SUKAMISKIN,MA,PENDIDIKAN UMUM,SWASTA,PENDIDIKAN MENENGAH,KOTA BANDUNG,KEC. ARCAMANIK,SUKAMISKIN,JL. RAYA TIMUR NO.128 KM.8 RT.01 RW.04,FORMAL,KEMENTERIAN AGAMA
551976,60707580,MIS ASSYAFIIYAH,MI,PENDIDIKAN UMUM,SWASTA,PENDIDIKAN DASAR,KAB. CIANJUR,KEC. CILAKU,CIHARASHAS,KP. CIRENYOM,FORMAL,KEMENTERIAN AGAMA


In [21]:
detail_list = [ get_school_details(n) for n in df['NPSN'] ] 
detail_list

Failed NPSN 69729399: Status 404
Failed NPSN 69729398: Status 404
Failed NPSN 60709932: Status 404
Failed NPSN 69887396: Status 404
Failed NPSN 69927574: Status 404
Failed NPSN 70034028: Status 404
Failed NPSN 20277574: Status 404
Failed NPSN 69750980: Status 404
Failed NPSN 69898526: Status 404
Failed NPSN 40104425: Status 404
Failed NPSN 70050136: Status 404
Failed NPSN 70046205: Status 404
Failed NPSN 69738103: Status 404
Failed NPSN 69736397: Status 404
Failed NPSN 70051007: Status 404
Failed NPSN 60703935: Status 404
Failed NPSN 60721779: Status 404
Failed NPSN 69854376: Status 404
Failed NPSN 69733480: Status 404
Failed NPSN 69752133: Status 404
Failed NPSN 60706827: Status 404
Failed NPSN 70044270: Status 404
Failed NPSN 10264181: Status 404
Failed NPSN 69888116: Status 404
Failed NPSN 69731562: Status 404
Failed NPSN 70026077: Status 404
Failed NPSN 60707061: Status 404
Failed NPSN 20280507: Status 404
Failed NPSN 69737471: Status 404
Failed NPSN 69958152: Status 404
Failed NPS

KeyboardInterrupt: 

In [ ]:
import sqlite3
import pandas as pd

# 1. Convert your list of dictionaries to a DataFrame
# Filter out None values just in case some API calls failed
df_details = pd.DataFrame([res for res in detail_list if res is not None])

# 2. Create a connection to the SQLite database
# If the file doesn't exist, it will be created automatically in your current directory
conn = sqlite3.connect('sekolah_data.db')

try:
    # 3. Store the DataFrame into a table named 'geolocation'
    # if_exists='replace' will overwrite the table
    # if_exists='append' will add new rows to existing ones
    df_details.to_sql('geolocation', conn, if_exists='replace', index=False)
    print("Successfully stored data to SQLite table: 'geolocation'")
except Exception as e:
    print(f"Error storing to database: {e}")
finally:
    # Always close the connection
    conn.close()